In [1]:
from glob import glob
import os
from tqdm import tqdm
from PIL import Image
from utils import rajouter_features

In [2]:
crop_size = 32
split = "val"   # choisir train, val ou test

root_dir = f"data/dataset_crops_paires/{split}"
os.makedirs(f"{root_dir}/sat", exist_ok=True)   # Création du dossier pour les images satellites si besoin
os.makedirs(f"{root_dir}/ann", exist_ok=True)   # Création du dossier pour les annotations si besoin

In [3]:
# get all image paths
img_paths = []

for path in glob(os.path.join(f"data/dataset/{split}", "**", f"*S2L2A*"), recursive=True):
    if path.lower().endswith((".jpg", ".jpeg", ".png")):

        # chemin vers l'image satellite
        img_path = path

        # chemin vers le masque de brûlé
        path_to_img_gra = img_path.replace("S2L2A", "GRA")
        path_to_img_del = img_path.replace("S2L2A", "DEL")
        if os.path.exists(path_to_img_gra):
            img_path_burn = path_to_img_gra
        elif os.path.exists(path_to_img_del):
            img_path_burn = path_to_img_del
        else:
            print("********** image brûlée non trouvée *********")
            continue

        # chemin vers l'image annotée
        img_ann_path = img_path.replace("S2L2A", "Esri10_LC")
        assert os.path.exists(img_ann_path), f"Annotation image not found: {img_ann_path}"

        img_paths.append((img_path, img_path_burn, img_ann_path))


print(len(img_paths), "images trouvées")

53 images trouvées


In [4]:
idx = 0

for i, (img_path, img_path_burn, img_ann_path) in tqdm(enumerate(img_paths), total=len(img_paths)):

    img = Image.open(img_path).convert("RGB")
    img_ann = Image.open(img_ann_path).convert("RGB")
    img_burn = Image.open(img_path_burn).convert("RGB")
    img_ann = rajouter_features(img_ann, img_burn)   # rajouter les features artificielles aux annotations
    assert img.size == img_ann.size, f"Image size mismatch: {img_path} and {img_ann_path}"


    width, height = img.size

    # Calculer combien de crops on peut faire horizontalement et verticalement
    n_crops_x = width // crop_size
    n_crops_y = height // crop_size

    # Découper en crops 32x32
    for i in range(n_crops_y):
        for j in range(n_crops_x):
            left = j * crop_size
            upper = i * crop_size
            right = left + crop_size
            lower = upper + crop_size

            crop = img.crop((left, upper, right, lower))
            crop_ann = img_ann.crop((left, upper, right, lower))

            crop.save(f"{root_dir}/sat/crop_{idx}.png")
            crop_ann.save(f"{root_dir}/ann/crop_{idx}.png")

            idx += 1


100%|██████████| 53/53 [03:21<00:00,  3.80s/it]
